In [1]:
# Install PyTorch/XLA (TPU backend)
# !pip install cloud-tpu-client==0.10 \
#     torch==2.8.0 torchvision \
#     https://storage.googleapis.com/libtpu-releases/index.html
# # The index URL goes in -f, not as a package
# !pip install torch_xla[tpu]==2.8.0 \
#     -f https://storage.googleapis.com/libtpu-releases/index.html

# Install HuggingFace stack
!pip install transformers datasets accelerate

In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import AutoModelForMaskedLM

model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [4]:
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> DistilBERT number of parameters: {round(distilbert_num_parameters)}M'")
print(f"'>>> BERT number of parameters: 110M'")

'>>> DistilBERT number of parameters: 67M'
'>>> BERT number of parameters: 110M'


In [5]:
text = "This is a great [MASK]"

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
import torch

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great !'
'>>> This is a great .'
'>>> This is a great deal'
'>>> This is a great adventure'
'>>> This is a great ;'


In [8]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
imdb_dataset

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [9]:
sample = imdb_dataset["train"].shuffle(seed=42).select(range(5))

for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...'
'>>> Label: 1'

'>>> Review: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stu

In [10]:
sample_unsupervised = imdb_dataset["unsupervised"].shuffle(seed=42).select(range(5))

for row in sample_unsupervised:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: If you've seen the classic Roger Corman version starring Vincent Price it's hard to put it out of your head, but you probably should do because this one is totally different. Subtlety has been abandoned in favour of gross-out horror - nudity, gore and all-round unpleasantness. OK it's ridiculous, trashy, sensationalised and historically dubious (did any members of the Inquisition really wear horn-rimmed glasses?), but despite all this it is strangely compelling. I literally couldn't tear myself away from the screen until the end of the movie. If there's a bigger compliment you can pay to a film I don't know what it is.'
'>>> Label: -1'

'>>> Review: For me, this was the most moving film of the decade. Samira Makhmalbaf shows pure bravery and vision in the making. She has an intelligence and gift for speaking to the people, regardless of their nationality or beliefs. I am inspired and touched by her humanity and can only hope that she has touched many people the same way. 

In [11]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched = True, remove_columns = ["text", "label"]
)
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

In [12]:
tokenizer.model_max_length

512

In [13]:
chunk_size = 128

In [14]:
tokenized_samples = tokenized_datasets["train"][:5]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> review {idx} length: {len(sample)}'")

'>>> review 0 length: 363'
'>>> review 1 length: 304'
'>>> review 2 length: 133'
'>>> review 3 length: 185'
'>>> review 4 length: 495'


In [15]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}

total_length = len(concatenated_examples["input_ids"])
print(f"'>>> Concatenated review length {total_length}'")

'>>> Concatenated review length 1480'


In [16]:
chunks = {
    k: [ t[i:i+chunk_size] for i in range(0, total_length, chunk_size)]
    for k,t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length : {len(chunk)}'")

'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 128'
'>>> Chunk length : 72'


In [17]:
def group_text(examples):
    # Concatenate all text
    concatenated_examples = {
        k: sum(examples[k],[]) for k in examples.keys()
    }

    # Compute length of the concatenated texts
    total_length = len(concatenated_examples["input_ids"])

    # Drop the last chunk if the its smaller
    total_length = (total_length // chunk_size) * chunk_size

    # Split the chunks

    chunks = {
        k: [t[i:i+chunk_size] for i in range(0, total_length, chunk_size)]
        for k,t in concatenated_examples.items()
    }

    # create new label columns why???

    chunks["labels"] = chunks["input_ids"].copy()
    return chunks

In [18]:
lm_datasets = tokenized_datasets.map(group_text, batched = True)
lm_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [19]:
tokenizer.decode(lm_datasets["train"][0]["labels"])

'[CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by u. s. customs if it ever tried to enter this country, therefore being a fan of films considered " controversial " i really had to see this for myself. < br / > < br / > the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life. in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such'

In [20]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability= 0.15)

In [21]:
# sample runnning the collator
sample_collator = [lm_datasets["train"][i] for i in range(2)]

for sample in sample_collator:
     _ = sample.pop("word_ids")

for chunk in data_collator(sample_collator)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious [unused752] yellow from [MASK] video store because of all the controversy that surrounded it when [MASK] was first released [MASK] [MASK]. i also heard that [MASK] first [MASK] was seized by u [MASK] s. customs if it ever tried to enter [MASK] country, [MASK] being [MASK] [MASK] of films considered " controversial " i really had to [MASK] this for [MASK]. < br / > < [MASK] / > [MASK] plot is centered around a young swedish drama student named lena [MASK] wants to learn everything she [MASK] about [MASK]. in particular [MASK] wants to focus her attention [MASK] to making some sort of documentary on what the average swede thought about certain political issues such'

'>>> as the vietnam war and race issues in the united states. in [MASK] asking [MASK] [MASK] ordinary denizens of stockholm about their opinions on politics, she has sex with [MASK] drama teacher [MASK] classmates [MASK] and married men. < [MASK] / > < br / > what kills me about i am curious

In [22]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probability = 0.2

# features here is a list of dicts representing individual batches
def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # Creating mapping table.
        mapping = collections.defaultdict(list)

        current_word  = None
        current_word_idx = -1

        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_idx += 1
                    mapping[current_word_idx].append(idx)

        # Randomly mask words

        mask = np.random.binomial(1, wwm_probability, (len(mapping),))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)

        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)



In [23]:
samples_wwm_collat = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples_wwm_collat)

for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - [MASK] from [MASK] video store because [MASK] all [MASK] controversy that surrounded it when [MASK] was first released in 1967 [MASK] i also heard that at first it was seized by u [MASK] s. customs if it [MASK] tried to enter this [MASK], therefore being a fan [MASK] films considered [MASK] [MASK] " i [MASK] had to see this [MASK] myself [MASK] < br / > < [MASK] / > the plot is centered [MASK] a young swedish drama student named lena who wants to learn everything she can about life. in [MASK] she wants to focus [MASK] [MASK]s [MASK] making some [MASK] of documentary on what the average swede [MASK] about certain political [MASK] such'

'>>> as the vietnam war and race issues in the united states. in [MASK] asking politicians and ordinary [MASK]izens of stockholm about [MASK] opinions on politics, she has sex with her drama teacher, classmates, and married men. < [MASK] [MASK] > < [MASK] / [MASK] what [MASK] me about i am curious - yellow [MASK] that 

In [24]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)

downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

In [25]:
!pip install accelerate

In [ ]:
from huggingface_hub import login
login(token="hf_XXX")

In [27]:
from transformers import TrainingArguments

batch_size = 64

logging_steps = len(downsampled_dataset["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps
)

In [28]:
from transformers import Trainer

# tokenizer is renamed to processing_class in the trainer in newer version of transformer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)


In [29]:
import math

eval_results_before_training = trainer.evaluate()

print(f">>> Perplexity: {math.exp(eval_results_before_training['eval_loss']):.2f}")

>>> Perplexity: 21.94


In [30]:
# training the model on non whole word mask
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.681356,2.492953,0.005800
2,2.582544,2.448033,0.005800
3,2.525844,2.482286,0.005800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.5959746265613854, metrics={'train_runtime': 179.1614, 'train_samples_per_second': 167.447, 'train_steps_per_second': 2.629, 'total_flos': 994208670720000.0, 'train_loss': 2.5959746265613854, 'epoch': 3.0})

In [31]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 12.05


In [32]:
# trainer.push_to_hub()

In [33]:
training_args_wwm = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
    remove_unused_columns=False
)

trainer_wwm = Trainer(
    model = model,
    args = training_args_wwm,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=whole_word_masking_data_collator,
    processing_class=tokenizer
)

In [34]:
trainer_wwm.train()

Epoch,Training Loss,Validation Loss
1,2.914283,2.798629
2,2.852993,2.817139
3,2.823007,2.783017


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.8636340163569036, metrics={'train_runtime': 183.9256, 'train_samples_per_second': 163.109, 'train_steps_per_second': 2.561, 'total_flos': 994208670720000.0, 'train_loss': 2.8636340163569036, 'epoch': 3.0})

In [35]:
eval_results = trainer_wwm.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 15.82


In [36]:
# Now we redo everything but with accelerator and fixed evals. for fixed evals, we will use separate dataLoaders for train and test/eval

def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

In [38]:
downsampled_dataset = downsampled_dataset.remove_columns("word_ids")

eval_dataset = downsampled_dataset['test'].map(
    insert_random_mask,
    batched=True,
    remove_columns=downsampled_dataset["test"].column_names
)

eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
        "masked_token_type_ids": "token_type_ids",
    }
)

eval_dataset = eval_dataset.remove_columns(["token_type_ids"])


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [39]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

batch_size = 64
train_dataloader = DataLoader(
    downsampled_dataset["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)

In [40]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [41]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [42]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [43]:
from transformers import get_scheduler

num_train_epochs = 3
num_updates_steps_per_epcoh = len(train_dataloader)
num_training_steps = num_train_epochs * num_updates_steps_per_epcoh

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)


In [44]:
from huggingface_hub import get_full_repo_name

model_name = "distilbert-base-uncased-finetuned-imdb-accelerate"
repo_name = get_full_repo_name(model_name)
output_dir = "distilbert-base-uncased-finetuned-imdb-accelerate"

In [45]:
from tqdm.auto import tqdm
import torch
import math

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
  # to swicth model to training mode
  model.train()
  for batch in train_dataloader:
    outputs = model(**batch)
    loss = outputs.loss
    accelerator.backward(loss)

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)

  model.eval()
  losses=[]
  for step, batch in enumerate(eval_dataloader):
    with torch.no_grad():
      outputs = model(**batch)

    eval_loss = outputs.loss
    losses.append(accelerator.gather(eval_loss.repeat(batch_size)))

  losses = torch.cat(losses)
  losses = losses[: len(eval_dataset)]
  try:
      perplexity = math.exp(torch.mean(losses))
  except OverflowError:
      perplexity = float("inf")

  print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

  0%|          | 0/471 [00:00<?, ?it/s]

>>> Epoch 0: Perplexity: 11.342366903495941
>>> Epoch 1: Perplexity: 10.847126059140619
>>> Epoch 2: Perplexity: 10.685367317616963


In [ ]:
print(eval_dataset.column_names)
# If you see 'masked_input_ids' instead of 'input_ids' → rename didn't apply


In [46]:
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.push_to_hub(repo_name, commit_message="Training complete")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pddm5pe/model.safetensors:   0%|          | 1.08MB /  268MB            

CommitInfo(commit_url='https://huggingface.co/jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate/commit/bc036e5f91c99b2baee5ed87a9f622b332252d72', commit_message='Training complete', commit_description='', oid='bc036e5f91c99b2baee5ed87a9f622b332252d72', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate', endpoint='https://huggingface.co', repo_type='model', repo_id='jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate'), pr_revision=None, pr_num=None)

In [51]:
# Push model directly to hub
tokenizer.push_to_hub(repo_name, commit_message="tokenizer push")

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate/commit/275b21fff358985442edd90e3eb03a1c91ee6572', commit_message='tokenizer push', commit_description='', oid='275b21fff358985442edd90e3eb03a1c91ee6572', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate', endpoint='https://huggingface.co', repo_type='model', repo_id='jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate'), pr_revision=None, pr_num=None)

In [52]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="jagarwal1999/distilbert-base-uncased-finetuned-imdb-accelerate")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [53]:
text = "This is a great [MASK]."
preds = pipe(text)

for pred in preds:
    print(f">>> {pred['sequence']}")

>>> this is a great film.
>>> this is a great movie.
>>> this is a great idea.
>>> this is a great one.
>>> this is a great adventure.
